# 66. The Workforce Scheduling for Peak/Off-Peak Problem
## Tier 3: The Advanced Algorithm (Grey Wolf Optimizer)

### Key assumptions
- Population-based metaheuristic can explore solution space effectively
- Social hierarchy (alpha, beta, delta) guides search toward optimal solutions
- Wolf positions represent complete workforce schedules
- Hunting behavior simulates optimization process

### Approach (step-by-step)
1. **Initialize wolf pack**: Create random population of schedule solutions
2. **Establish social hierarchy**: Identify alpha, beta, and delta (best 3 solutions)
3. **Position update equations**: Guide omega wolves toward leaders
4. **Exploration-exploitation balance**: Parameter 'a' controls search behavior
5. **Iterative hunting**: Update positions and converge to optimal solution

### What to look for in the results
- Convergence behavior of the wolf pack
- Alpha, beta, delta solution quality over time
- Population diversity and exploration patterns
- Final schedule quality vs computational time

### Concrete example (from the source)
Grey Wolf Optimizer with hunting behavior:
- **Alpha (α)**: Best solution found so far
- **Beta (β)**: Second-best solution
- **Delta (δ)**: Third-best solution
- **Omega (ω)**: Rest of the pack following leaders
- **Position updates**: Based on distance to alpha, beta, delta wolves

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
random.seed(42)

In [ ]:
# Define the Grey Wolf Optimizer for workforce scheduling
class GreyWolfOptimizer:
    """Grey Wolf Optimizer for workforce scheduling"""
    
    def __init__(self, employees: List[Dict], time_periods: List[int], 
                 demand: List[int], cost_params: Dict, 
                 pack_size: int = 20, max_iterations: int = 100):
        """
        Initialize Grey Wolf Optimizer
        
        Args:
            employees: List of employee dictionaries
            time_periods: List of time period indices
            demand: Required staff for each time period
            cost_params: Cost parameters
            pack_size: Number of wolves in the pack
            max_iterations: Maximum number of iterations
        """
        self.employees = employees
        self.time_periods = time_periods
        self.demand = demand
        self.cost_params = cost_params
        self.pack_size = pack_size
        self.max_iterations = max_iterations
        
        # GWO parameters
        self.a_initial = 2.0  # Initial control parameter
        self.a_final = 0.0    # Final control parameter
        
        # Initialize wolf pack
        self.pack = []
        self.alpha = None
        self.beta = None
        self.delta = None
        
        # Track convergence
        self.convergence_history = []
        self.diversity_history = []
        
    def encode_schedule(self, schedule: Dict) -> np.ndarray:
        """
        Encode schedule as position vector for GWO
        
        Args:
            schedule: Schedule dictionary
        
        Returns:
            Position vector
        """
        # Create binary matrix: employees x time_periods
        position = np.zeros((len(self.employees), len(self.time_periods)))
        
        for emp_idx, emp in enumerate(self.employees):
            emp_id = emp['id']
            if emp_id in schedule:
                for period in schedule[emp_id]:
                    if 0 <= period < len(self.time_periods):
                        position[emp_idx, period] = 1
        
        return position.flatten()
    
    def decode_schedule(self, position: np.ndarray) -> Dict:
        """
        Decode position vector back to schedule
        
        Args:
            position: Position vector
        
        Returns:
            Schedule dictionary
        """
        schedule = {}
        position_matrix = position.reshape((len(self.employees), len(self.time_periods)))
        
        for emp_idx, emp in enumerate(self.employees):
            emp_id = emp['id']
            schedule[emp_id] = []
            
            for period_idx in range(len(self.time_periods)):
                if position_matrix[emp_idx, period_idx] > 0.5:  # Binary threshold
                    schedule[emp_id].append(period_idx)
        
        return schedule
    
    def generate_random_schedule(self) -> Dict:
        """
        Generate a random feasible schedule
        
        Returns:
            Random schedule dictionary
        """
        schedule = {emp['id']: [] for emp in self.employees}
        
        # For each time period, assign random employees
        for t in range(len(self.demand)):
            required = self.demand[t]
            available_employees = [emp for emp in self.employees 
                                 if self.is_employee_available(emp, t, schedule)]
            
            if available_employees:
                # Randomly select required number of employees
                selected = random.sample(available_employees, 
                                       min(required, len(available_employees)))
                for emp in selected:
                    schedule[emp['id']].append(t)
        
        return schedule
    
    def is_employee_available(self, employee: Dict, period: int, 
                             schedule: Dict) -> bool:
        """
        Check if employee is available for a time period
        
        Args:
            employee: Employee dictionary
            period: Time period to check
            schedule: Current schedule
        
        Returns:
            True if employee is available
        """
        emp_id = employee['id']
        current_assignments = len(schedule.get(emp_id, []))
        max_hours = employee.get('max_hours', 8)
        
        return current_assignments < max_hours
    
    def calculate_fitness(self, schedule: Dict) -> float:
        """
        Calculate fitness (inverse of cost) for a schedule
        
        Args:
            schedule: Schedule dictionary
        
        Returns:
            Fitness value (higher is better)
        """
        cost = self.calculate_schedule_cost(schedule)
        
        # Add penalty for infeasible schedules
        penalty = 0
        for t in range(len(self.demand)):
            required = self.demand[t]
            assigned = sum(1 for periods in schedule.values() if t in periods)
            if assigned < required:
                penalty += (required - assigned) * 1000  # Large penalty for understaffing
        
        total_cost = cost + penalty
        
        # Return fitness (inverse of cost, avoiding division by zero)
        return 1.0 / (1.0 + total_cost)
    
    def calculate_schedule_cost(self, schedule: Dict) -> float:
        """
        Calculate total cost of a schedule
        
        Args:
            schedule: Schedule dictionary
        
        Returns:
            Total cost
        """
        total_cost = 0
        
        # Labor cost
        for emp_id, periods in schedule.items():
            emp = next(e for e in self.employees if e['id'] == emp_id)
            hourly_rate = emp.get('hourly_rate', self.cost_params['default_labor_cost'])
            total_cost += len(periods) * hourly_rate
        
        # Understaffing and overstaffing costs
        for t in range(len(self.demand)):
            required = self.demand[t]
            assigned = sum(1 for periods in schedule.values() if t in periods)
            
            if assigned < required:
                total_cost += (required - assigned) * self.cost_params['understaff_penalty']
            elif assigned > required:
                total_cost += (assigned - required) * self.cost_params['overstaff_penalty']
        
        return total_cost
    
    def initialize_pack(self):
        """
        Initialize the wolf pack with random schedules
        """
        self.pack = []
        
        for _ in range(self.pack_size):
            schedule = self.generate_random_schedule()
            position = self.encode_schedule(schedule)
            fitness = self.calculate_fitness(schedule)
            
            wolf = {
                'position': position,
                'schedule': schedule,
                'fitness': fitness,
                'cost': self.calculate_schedule_cost(schedule)
            }
            
            self.pack.append(wolf)
        
        # Sort by fitness and establish hierarchy
        self.pack.sort(key=lambda w: w['fitness'], reverse=True)
        
        if len(self.pack) >= 3:
            self.alpha = self.pack[0]
            self.beta = self.pack[1]
            self.delta = self.pack[2]
        elif len(self.pack) >= 2:
            self.alpha = self.pack[0]
            self.beta = self.pack[1]
            self.delta = self.pack[0]
        else:
            self.alpha = self.pack[0]
            self.beta = self.pack[0]
            self.delta = self.pack[0]
    
    def update_hierarchy(self):
        """
        Update alpha, beta, and delta wolves
        """
        # Sort pack by fitness
        self.pack.sort(key=lambda w: w['fitness'], reverse=True)
        
        # Update leaders
        self.alpha = self.pack[0]
        if len(self.pack) >= 2:
            self.beta = self.pack[1]
        else:
            self.beta = self.pack[0]
            
        if len(self.pack) >= 3:
            self.delta = self.pack[2]
        else:
            self.delta = self.pack[0]
    
    def calculate_diversity(self) -> float:
        """
        Calculate population diversity
        
        Returns:
            Diversity measure
        """
        if len(self.pack) < 2:
            return 0.0
        
        positions = np.array([wolf['position'] for wolf in self.pack])
        mean_position = np.mean(positions, axis=0)
        
        # Calculate average distance from mean
        distances = [np.linalg.norm(pos - mean_position) for pos in positions]
        return np.mean(distances)
    
    def hunting_behavior(self, iteration: int):
        """
        Perform hunting behavior - update omega wolf positions
        
        Args:
            iteration: Current iteration number
        """
        # Update control parameter 'a' (linearly decreasing)
        a = self.a_initial - (self.a_initial - self.a_final) * (iteration / self.max_iterations)
        
        # Update positions of omega wolves (excluding alpha, beta, delta)
        for wolf_idx in range(3, len(self.pack)):
            wolf = self.pack[wolf_idx]
            current_pos = wolf['position']
            new_pos = np.zeros_like(current_pos)
            
            # Update position based on alpha, beta, and delta
            for dim in range(len(current_pos)):
                # Distance to alpha
                r1_alpha = random.random()
                r2_alpha = random.random()
                A_alpha = 2 * a * r1_alpha - a
                C_alpha = 2 * r2_alpha
                D_alpha = abs(C_alpha * self.alpha['position'][dim] - current_pos[dim])
                X1_alpha = self.alpha['position'][dim] - A_alpha * D_alpha
                
                # Distance to beta
                r1_beta = random.random()
                r2_beta = random.random()
                A_beta = 2 * a * r1_beta - a
                C_beta = 2 * r2_beta
                D_beta = abs(C_beta * self.beta['position'][dim] - current_pos[dim])
                X1_beta = self.beta['position'][dim] - A_beta * D_beta
                
                # Distance to delta
                r1_delta = random.random()
                r2_delta = random.random()
                A_delta = 2 * a * r1_delta - a
                C_delta = 2 * r2_delta
                D_delta = abs(C_delta * self.delta['position'][dim] - current_pos[dim])
                X1_delta = self.delta['position'][dim] - A_delta * D_delta
                
                # Average position update
                new_pos[dim] = (X1_alpha + X1_beta + X1_delta) / 3.0
                
                # Apply bounds (binary constraint)
                new_pos[dim] = np.clip(new_pos[dim], 0, 1)
            
            # Update wolf position
            wolf['position'] = new_pos
            wolf['schedule'] = self.decode_schedule(new_pos)
            wolf['fitness'] = self.calculate_fitness(wolf['schedule'])
            wolf['cost'] = self.calculate_schedule_cost(wolf['schedule'])
    
    def optimize(self) -> Dict:
        """
        Run Grey Wolf Optimizer
        
        Returns:
            Optimization results
        """
        print(f"Starting Grey Wolf Optimizer with {self.pack_size} wolves for {self.max_iterations} iterations")
        
        # Initialize pack
        self.initialize_pack()
        
        print(f"Initial best cost: ${self.alpha['cost']:.2f}")
        
        # Optimization loop
        for iteration in range(self.max_iterations):
            # Hunting behavior - update omega wolves
            self.hunting_behavior(iteration)
            
            # Update hierarchy
            self.update_hierarchy()
            
            # Record convergence
            self.convergence_history.append({
                'iteration': iteration,
                'alpha_cost': self.alpha['cost'],
                'beta_cost': self.beta['cost'],
                'delta_cost': self.delta['cost'],
                'avg_cost': np.mean([w['cost'] for w in self.pack])
            })
            
            # Record diversity
            diversity = self.calculate_diversity()
            self.diversity_history.append(diversity)
            
            # Print progress
            if (iteration + 1) % 10 == 0:
                print(f"Iteration {iteration + 1}: Best cost = ${self.alpha['cost']:.2f}, "
                      f"Diversity = {diversity:.4f}")
        
        print(f"\nOptimization completed!")
        print(f"Final best cost: ${self.alpha['cost']:.2f}")
        print(f"Total improvement: ${self.convergence_history[0]['alpha_cost'] - self.alpha['cost']:.2f}")
        
        return {
            'best_schedule': self.alpha['schedule'],
            'best_cost': self.alpha['cost'],
            'convergence_history': self.convergence_history,
            'diversity_history': self.diversity_history,
            'final_pack': self.pack
        }

In [ ]:
# Create test data for GWO
employees = [
    {'id': 1, 'name': 'Alice', 'hourly_rate': 22, 'max_hours': 8},
    {'id': 2, 'name': 'Bob', 'hourly_rate': 20, 'max_hours': 8},
    {'id': 3, 'name': 'Charlie', 'hourly_rate': 24, 'max_hours': 6},
    {'id': 4, 'name': 'Diana', 'hourly_rate': 21, 'max_hours': 8},
    {'id': 5, 'name': 'Eve', 'hourly_rate': 19, 'max_hours': 8},
    {'id': 6, 'name': 'Frank', 'hourly_rate': 23, 'max_hours': 8}
]

time_periods = [0, 1, 2, 3, 4, 5, 6, 7]  # 8 time periods
demand = [1, 2, 4, 3, 2, 3, 2, 1]  # Variable demand pattern

cost_params = {
    'default_labor_cost': 20,
    'understaff_penalty': 50,
    'overstaff_penalty': 10
}

print("Grey Wolf Optimizer Test Data:")
print(f"Employees: {len(employees)}")
print(f"Time periods: {len(time_periods)}")
print(f"Demand pattern: {demand}")
print(f"Total demand: {sum(demand)} staff-periods")

In [ ]:
# Initialize and run GWO
gwo = GreyWolfOptimizer(
    employees=employees,
    time_periods=time_periods,
    demand=demand,
    cost_params=cost_params,
    pack_size=20,
    max_iterations=100
)

# Run optimization
start_time = time.time()
results = gwo.optimize()
end_time = time.time()

print(f"\nOptimization time: {end_time - start_time:.2f} seconds")
print(f"Final best cost: ${results['best_cost']:.2f}")

In [ ]:
# Analyze the final schedule
best_schedule = results['best_schedule']

print("Final Schedule from Alpha Wolf:")
schedule_summary = []

for emp_id, periods in best_schedule.items():
    emp = next(e for e in employees if e['id'] == emp_id)
    if periods:  # Only show employees with assignments
        schedule_summary.append({
            'employee': emp['name'],
            'periods': periods,
            'hours_worked': len(periods),
            'cost': len(periods) * emp['hourly_rate']
        })

schedule_df = pd.DataFrame(schedule_summary)
print(schedule_df.to_string(index=False))

print(f"\nTotal labor cost: ${schedule_df['cost'].sum():.2f}")

# Check demand satisfaction
print("\nDemand Satisfaction:")
total_understaff = 0
total_overstaff = 0

for t in range(len(demand)):
    required = demand[t]
    assigned = sum(1 for periods in best_schedule.values() if t in periods)
    status = "✓" if assigned >= required else "✗"
    understaff = max(0, required - assigned)
    overstaff = max(0, assigned - required)
    
    total_understaff += understaff
    total_overstaff += overstaff
    
    print(f"Period {t+1}: Required {required}, Assigned {assigned} {status}")

print(f"\nTotal understaff periods: {total_understaff}")
print(f"Total overstaff periods: {total_overstaff}")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Grey Wolf Optimizer Analysis', fontsize=16, fontweight='bold')

# 1. Convergence Plot
ax1 = axes[0, 0]
history = results['convergence_history']
iterations = [h['iteration'] for h in history]
alpha_costs = [h['alpha_cost'] for h in history]
beta_costs = [h['beta_cost'] for h in history]
delta_costs = [h['delta_cost'] for h in history]
avg_costs = [h['avg_cost'] for h in history]

ax1.plot(iterations, alpha_costs, 'r-', linewidth=2, label='Alpha (Best)')
ax1.plot(iterations, beta_costs, 'b-', linewidth=1.5, label='Beta (2nd Best)')
ax1.plot(iterations, delta_costs, 'g-', linewidth=1.5, label='Delta (3rd Best)')
ax1.plot(iterations, avg_costs, 'k--', linewidth=1, alpha=0.7, label='Pack Average')

ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cost ($)')
ax1.set_title('Convergence of Wolf Pack')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Population Diversity
ax2 = axes[0, 1]
diversity = results['diversity_history']
ax2.plot(iterations, diversity, 'purple', linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Population Diversity')
ax2.set_title('Population Diversity Over Time')
ax2.grid(True, alpha=0.3)

# 3. Demand vs Staff Coverage
ax3 = axes[1, 0]
periods = list(range(len(demand)))
coverage_vals = [sum(1 for periods in best_schedule.values() if t in periods) 
                 for t in periods]

ax3.plot(periods, demand, 'ro-', label='Demand', linewidth=2, markersize=8)
ax3.plot(periods, coverage_vals, 'bs-', label='Staff Coverage', linewidth=2, markersize=8)
ax3.set_xlabel('Time Period')
ax3.set_ylabel('Number of Staff')
ax3.set_title('Demand vs Staff Coverage')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xticks(periods)

# 4. Final Pack Fitness Distribution
ax4 = axes[1, 1]
final_pack = results['final_pack']
costs = [wolf['cost'] for wolf in final_pack]
fitness_values = [wolf['fitness'] for wolf in final_pack]

# Highlight alpha, beta, delta
colors = ['red' if i == 0 else 'blue' if i == 1 else 'green' if i == 2 else 'gray' 
         for i in range(len(final_pack))]

scatter = ax4.scatter(range(len(final_pack)), costs, c=colors, s=100, alpha=0.7)
ax4.set_xlabel('Wolf Rank')
ax4.set_ylabel('Cost ($)')
ax4.set_title('Final Pack Cost Distribution')
ax4.grid(True, alpha=0.3)

# Add legend for leaders
ax4.scatter([], [], c='red', s=100, label='Alpha')
ax4.scatter([], [], c='blue', s=100, label='Beta')
ax4.scatter([], [], c='green', s=100, label='Delta')
ax4.scatter([], [], c='gray', s=100, label='Omega')
ax4.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate the hunting behavior mechanics
print("Demonstrating Grey Wolf Hunting Behavior:")
print("=" * 50)

# Create a simple example with smaller pack
gwo_demo = GreyWolfOptimizer(
    employees=employees[:4],  # Use 4 employees for simplicity
    time_periods=time_periods[:6],  # Use 6 time periods
    demand=demand[:6],
    cost_params=cost_params,
    pack_size=8,
    max_iterations=5
)

# Initialize and show initial state
gwo_demo.initialize_pack()

print(f"Initial Alpha cost: ${gwo_demo.alpha['cost']:.2f}")
print(f"Initial Beta cost: ${gwo_demo.beta['cost']:.2f}")
print(f"Initial Delta cost: ${gwo_demo.delta['cost']:.2f}")
print(f"Initial pack diversity: {gwo_demo.calculate_diversity():.4f}")

# Show one iteration of hunting behavior
print("\nIteration 1 - Hunting Behavior:")
print("Control parameter 'a':", gwo_demo.a_initial)

# Demonstrate position update for one omega wolf
if len(gwo_demo.pack) > 3:
    omega_idx = 3
    omega_before = gwo_demo.pack[omega_idx]['position'].copy()
    
    # Perform one hunting step
    gwo_demo.hunting_behavior(0)
    gwo_demo.update_hierarchy()
    
    omega_after = gwo_demo.pack[omega_idx]['position']
    
    # Calculate position change
    position_change = np.linalg.norm(omega_after - omega_before)
    
    print(f"Omega wolf {omega_idx} position change: {position_change:.4f}")
    print(f"Omega wolf cost before: ${gwo_demo.pack[omega_idx]['cost']:.2f}")
    
    # Show distance to leaders
    dist_to_alpha = np.linalg.norm(omega_after - gwo_demo.alpha['position'])
    dist_to_beta = np.linalg.norm(omega_after - gwo_demo.beta['position'])
    dist_to_delta = np.linalg.norm(omega_after - gwo_demo.delta['position'])
    
    print(f"Distance to Alpha: {dist_to_alpha:.4f}")
    print(f"Distance to Beta: {dist_to_beta:.4f}")
    print(f"Distance to Delta: {dist_to_delta:.4f}")

print(f"\nAfter hunting - Alpha cost: ${gwo_demo.alpha['cost']:.2f}")
print(f"New pack diversity: {gwo_demo.calculate_diversity():.4f}")

In [ ]:
# Compare different parameter settings
print("Parameter Sensitivity Analysis:")
print("=" * 40)

parameter_tests = [
    {'pack_size': 10, 'max_iterations': 50, 'name': 'Small Pack'},
    {'pack_size': 20, 'max_iterations': 100, 'name': 'Medium Pack'},
    {'pack_size': 30, 'max_iterations': 150, 'name': 'Large Pack'}
]

results_comparison = []

for test_params in parameter_tests:
    print(f"\nTesting {test_params['name']}:")
    
    gwo_test = GreyWolfOptimizer(
        employees=employees,
        time_periods=time_periods,
        demand=demand,
        cost_params=cost_params,
        pack_size=test_params['pack_size'],
        max_iterations=test_params['max_iterations']
    )
    
    start_time = time.time()
    test_result = gwo_test.optimize()
    end_time = time.time()
    
    results_comparison.append({
        'name': test_params['name'],
        'pack_size': test_params['pack_size'],
        'final_cost': test_result['best_cost'],
        'iterations': test_params['max_iterations'],
        'time': end_time - start_time,
        'initial_cost': test_result['convergence_history'][0]['alpha_cost'],
        'improvement': test_result['convergence_history'][0]['alpha_cost'] - test_result['best_cost']
    })
    
    print(f"  Final cost: ${test_result['best_cost']:.2f}")
    print(f"  Time: {end_time - start_time:.2f}s")
    print(f"  Improvement: ${results_comparison[-1]['improvement']:.2f}")

# Create comparison visualization
comparison_df = pd.DataFrame(results_comparison)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('GWO Parameter Sensitivity Analysis', fontsize=14, fontweight='bold')

# Cost comparison
ax1 = axes[0]
bars1 = ax1.bar(comparison_df['name'], comparison_df['final_cost'], 
                 color=['#2ecc71', '#3498db', '#f39c12'], alpha=0.8)
ax1.set_ylabel('Final Cost ($)')
ax1.set_title('Solution Quality')
ax1.grid(True, alpha=0.3)

for bar, cost in zip(bars1, comparison_df['final_cost']):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
             f'${cost:.0f}', ha='center', va='bottom')

# Time comparison
ax2 = axes[1]
bars2 = ax2.bar(comparison_df['name'], comparison_df['time'], 
                 color=['#e74c3c', '#9b59b6', '#34495e'], alpha=0.8)
ax2.set_ylabel('Time (seconds)')
ax2.set_title('Computational Time')
ax2.grid(True, alpha=0.3)

for bar, time_val in zip(bars2, comparison_df['time']):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
             f'{time_val:.1f}s', ha='center', va='bottom')

# Improvement comparison
ax3 = axes[2]
bars3 = ax3.bar(comparison_df['name'], comparison_df['improvement'], 
                 color=['#16a085', '#27ae60', '#2980b9'], alpha=0.8)
ax3.set_ylabel('Cost Improvement ($)')
ax3.set_title('Total Improvement')
ax3.grid(True, alpha=0.3)

for bar, improvement in zip(bars3, comparison_df['improvement']):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
             f'${improvement:.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nParameter Analysis Summary:")
print(comparison_df[['name', 'pack_size', 'final_cost', 'time', 'improvement']].to_string(index=False))

### Why this Tier exists vs earlier Tiers
The Grey Wolf Optimizer provides advanced capabilities beyond both MDP and VND:
- **Global optimization**: Population-based search escapes local optima better than VND
- **Nature-inspired intelligence**: Social hierarchy mimics successful natural systems
- **Balanced exploration**: Parameter 'a' automatically balances exploration vs exploitation
- **Parallel search**: Multiple wolves explore different solution regions simultaneously

### Pros / Cons vs earlier Tiers
**vs Tier 1 (MDP):**
- ✅ **Scales to much larger problems** without exponential complexity
- ✅ **Handles complex constraints** more flexibly
- ✅ **Better solution quality** for complex, non-linear problems
- ❌ **No optimality guarantee** unlike exact MDP solution

**vs Tier 2 (VND):**
- ✅ **Better global search** - less likely to get stuck in local optima
- ✅ **Parallel exploration** - multiple search directions simultaneously
- ✅ **Automatic parameter adaptation** - 'a' parameter controls search behavior
- ✅ **More robust** to different initial conditions
- ❌ **More computationally expensive** than VND for small problems
- ❌ **More complex to tune** and understand

### When to use this Tier
- **Large-scale scheduling problems** where VND gets stuck in local optima
- **Complex fitness landscapes** with many local optima
- **When solution quality is critical** and computational resources are available
- **Multi-modal problems** where different solution regions need exploration
- **Research and development** settings where advanced metaheuristics are valued
- **When you need robust performance** across different problem instances